In [ ]:
%pip install -q langchain langchain-aws langsmith boto3 python-dotenv

# Week 5 Monday -- LangSmith Observability & Tracing

Last week you **built agents** — you gave LLMs access to tools, wired up the agentic loop, and watched your agents reason about which tool to call, parse results, and generate responses. Your agents *work*.

But here is the hard question: **how do you know what is actually happening inside?**

When an agent picks the wrong tool, burns through tokens on an unnecessary loop, or returns a subtly wrong answer, where do you even begin debugging? Print statements? Guessing? Hope?

**LangSmith** is the answer. It is LangChain's observability platform — a service that automatically records every LLM call, every tool invocation, every token spent, and lays it all out in a visual trace you can click through step by step.

Today we go from *"it works… I think?"* to *"I know exactly what happened and why."*

## Learning Objectives

By the end of this lecture you will be able to:

1. **Configure LangSmith** for automatic tracing with three environment variables
2. **Read and interpret traces** — parent/child hierarchies, token counts, latency
3. **Explore traces in the dashboard** — filter, compare, and analyze agent runs
4. **Debug agent failures** — tool errors, wrong tool selection, and runaway loops

## Lecture Roadmap

| Stage | Topic | What You'll Do |
|-------|-------|----------------|
| **1** | LangSmith Setup & Configuration | Set env vars, verify tracing, navigate the dashboard |
| **2** | Trace Exploration & Analysis | Generate multi-tool traces, read hierarchies, track tokens |
| **3** | Debugging Agent Failures | Trigger errors, trace backwards, fix tool descriptions |

---

## Stage 1 — LangSmith Setup & Configuration

LangSmith requires exactly **three environment variables** to start capturing traces. Once those are set, every `create_agent` call, every tool invocation, and every LLM request is automatically recorded — no code changes needed.

| Variable | Required | Purpose |
|----------|----------|----------|
| `LANGSMITH_TRACING` | Yes | Set to `"true"` to enable automatic tracing |
| `LANGSMITH_API_KEY` | Yes | Your personal API key from [smith.langchain.com](https://smith.langchain.com) |
| `LANGSMITH_PROJECT` | No | Groups traces into a named project (defaults to `"default"`) |

> **Getting your API key:** Go to [smith.langchain.com](https://smith.langchain.com) → Profile → Settings → API Keys → Create API Key. Copy it immediately — you will not see it again. The key format looks like `lsv2_pt_xxxxxxxxxxxxxxxxxxxx`.

Put these in a `.env` file at your project root:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_pt_your_key_here
LANGSMITH_PROJECT=week5-monday-demo
```

> **Security:** Never commit `.env` files to version control. Add `.env` to your `.gitignore` immediately.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print("=" * 60)
print("Checking LangSmith Configuration")
print("=" * 60)

env_vars = {
    "LANGSMITH_TRACING": os.getenv("LANGSMITH_TRACING"),
    "LANGSMITH_API_KEY": os.getenv("LANGSMITH_API_KEY"),
    "LANGSMITH_PROJECT": os.getenv("LANGSMITH_PROJECT"),
}

all_set = True
for var, value in env_vars.items():
    if value:
        display = value[:8] + "..." if var == "LANGSMITH_API_KEY" and len(value) > 8 else value
        print(f"  ✓ {var} = {display}")
    else:
        print(f"  ✗ {var} = NOT SET")
        all_set = False

if not all_set:
    print("\nSome variables are missing. Create a .env file with:")
    print("  LANGSMITH_TRACING=true")
    print("  LANGSMITH_API_KEY=lsv2_pt_your_key_here")
    print("  LANGSMITH_PROJECT=week5-monday-demo")
else:
    print("\nAll LangSmith variables configured — tracing is active.")

Once those three variables are set, **every LangChain operation is automatically traced** — no decorators, no wrappers, no extra code. LangSmith captures:

- **Agent runs** — start time, end time, status, agent name
- **LLM calls** — prompt, response, model, token counts, latency
- **Tool calls** — tool name, input arguments, return value, duration
- **Errors** — exception type, message, and stack trace

```
Your Application                     LangSmith Dashboard
┌─────────────────┐                 ┌─────────────────────┐
│   Agent Code    │───traces───────▶│  Visual Execution   │
│                 │                 │  Timeline, Costs,   │
│  create_agent() │                 │  Debugging Tools    │
└─────────────────┘                 └─────────────────────┘
```

Let's verify that tracing works by creating a simple agent and invoking it. After running the cell below, you should see the trace appear in your [LangSmith dashboard](https://smith.langchain.com/).

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_greeting(name: str) -> str:
    """Generate a greeting for a person. Use when asked to greet someone."""
    return f"Hello, {name}! Welcome to the LangSmith tracing demo."

agent = create_agent(
    model="bedrock:us.amazon.nova-lite-v1:0",
    tools=[get_greeting],
    system_prompt="You are a friendly greeter. Use the greeting tool when asked to greet someone.",
    name="setup_verification_agent"
)

print("Invoking agent — this creates a trace in LangSmith...\n")

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please greet Alice"}]
})

print(f"User:  Please greet Alice")
print(f"Agent: {result['messages'][-1].content}")
print("\nOpen https://smith.langchain.com/ and find the trace for 'setup_verification_agent'.")

### What You Should See in the Dashboard

Open your LangSmith project and click into the trace. You will see a **tree structure** showing every step the agent took:

```
Run: setup_verification_agent (parent)
├── LLM Call (child)
│   ├── Input: system prompt + user message
│   └── Output: tool_call → get_greeting(name="Alice")
├── Tool: get_greeting (child)
│   ├── Input: {"name": "Alice"}
│   └── Output: "Hello, Alice! Welcome to..."
└── LLM Call (child)
    ├── Input: conversation + tool result
    └── Output: final response to user
```

Each node shows **duration** (how long that step took), **token counts** (input and output), and **status** (success or error). This is your X-ray into the agent's decision-making.

### Trace Properties at a Glance

| Component | What LangSmith Captures |
|-----------|-------------------------|
| **Agent Run** | Start time, end time, status, agent name |
| **LLM Calls** | Prompt, response, tokens, latency, model |
| **Tool Calls** | Tool name, arguments, return value, duration |
| **State Changes** | Messages added, state mutations |
| **Errors** | Exception type, message, stack trace |

> **Key insight:** Tracing is automatic. You did not add any tracing code — LangSmith hooks into LangChain under the hood. Every agent you have built (or will build) gets traced for free once those env vars are set.

---

## Stage 2 — Trace Exploration & Analysis

A single-tool trace is nice, but agents in the real world juggle multiple tools. That is where traces become genuinely useful — you can see *which* tools the agent chose, *in what order*, and *how long each step took*.

We will build a multi-tool business analyst agent, run it against three different queries of increasing complexity, and then compare the traces side by side in the dashboard.

In [ ]:
import time
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_database(query: str) -> str:
    """Search the company database for information about revenue, employees, or products."""
    time.sleep(0.5)
    results = {
        "revenue": "Q4 2025 revenue was $12.5M, up 15% year-over-year",
        "employees": "Current headcount is 450 employees across 3 offices",
        "products": "Main products: CloudSync, DataFlow, and AIAssist",
    }
    for key, value in results.items():
        if key in query.lower():
            return value
    return f"Search results for '{query}': No specific match found."

@tool
def calculate_metric(formula: str) -> str:
    """Calculate a business metric like growth rate, ratio, or percentage."""
    time.sleep(0.3)
    return f"Calculation result for '{formula}': 42.5%"

@tool
def generate_report(topic: str) -> str:
    """Generate a summary report on a business topic."""
    time.sleep(0.8)
    return f"Report on {topic}: Executive summary with key findings and recommendations."

business_agent = create_agent(
    model="bedrock:us.amazon.nova-lite-v1:0",
    tools=[search_database, calculate_metric, generate_report],
    system_prompt=(
        "You are a business analyst assistant. "
        "Use the available tools to answer questions about the company. "
        "Always use relevant tools before providing your final answer."
    ),
    name="business_analyst_agent"
)

print("Agent created: business_analyst_agent")
print("Tools: search_database, calculate_metric, generate_report")

In [ ]:
print("=" * 60)
print("Scenario 1: Simple query (single tool expected)")
print("=" * 60)
result1 = business_agent.invoke({
    "messages": [{"role": "user", "content": "What is our current employee count?"}]
})
print(f"Query:    What is our current employee count?")
print(f"Response: {result1['messages'][-1].content[:200]}")

print("\n" + "=" * 60)
print("Scenario 2: Complex query (multiple tools expected)")
print("=" * 60)
result2 = business_agent.invoke({
    "messages": [{"role": "user", "content": "Find our Q4 revenue and calculate the growth rate compared to last year."}]
})
print(f"Query:    Find Q4 revenue and calculate growth rate")
print(f"Response: {result2['messages'][-1].content[:200]}")

print("\n" + "=" * 60)
print("Scenario 3: Report generation")
print("=" * 60)
result3 = business_agent.invoke({
    "messages": [{"role": "user", "content": "Generate a report on our product portfolio."}]
})
print(f"Query:    Generate product portfolio report")
print(f"Response: {result3['messages'][-1].content[:200]}")

print("\nAll 3 traces are now in LangSmith. Open the dashboard and compare them.")

### Reading the Traces

Go to your LangSmith dashboard and find the three traces for `business_analyst_agent`. Here is what to look for:

**Scenario 1 (employee count)** — Expect a short trace: one LLM call deciding to use `search_database`, the tool execution, and a final LLM call generating the response. Three steps total.

**Scenario 2 (revenue + growth)** — Expect a longer trace: the LLM decides to call `search_database` for revenue data, *then* calls `calculate_metric` for the growth rate, and finally synthesizes a response. More steps, more tokens, longer duration.

**Scenario 3 (report)** — Expect a moderate trace: one tool call to `generate_report`, but the tool itself simulates a longer execution time (0.8s).

### Trace Hierarchy

Every trace forms a **tree** of parent-child relationships:

```
Run: business_analyst_agent (4.2s)
├── ChatBedrock (0.8s)
│   ├── Input: [SystemMessage, HumanMessage]
│   └── Output: AIMessage with tool_call
├── search_database (0.5s)
│   ├── Input: {"query": "revenue"}
│   └── Output: "Q4 2025 revenue was $12.5M..."
├── ChatBedrock (0.9s)
│   ├── Input: [SystemMessage, HumanMessage, AIMessage, ToolMessage]
│   └── Output: AIMessage (final response)
└── Total tokens: 847 (input: 623, output: 224)
```

### Token Usage

Click into any LLM call node and you will see input and output token counts. Notice how multi-tool traces use *more* tokens — each LLM call includes the full conversation history plus previous tool results.

This is why token monitoring matters: a single extra tool call doesn't just cost tool execution time, it also costs tokens for the LLM to process the growing context.

### Identifying Healthy vs. Warning Patterns

| Pattern | What It Looks Like | What It Means |
|---------|--------------------|---------------|
| **Healthy** | LLM → Tool → LLM → Response | Clean, linear execution |
| **Healthy** | LLM → Tool A + Tool B → LLM → Response | Efficient multi-tool use |
| **Warning** | LLM → LLM → LLM (no tools) | Agent may be confused |
| **Warning** | Same tool called 3+ times | Possible loop or unclear results |
| **Warning** | Very high token count | Inefficient prompt or tool response |

In [ ]:
def estimate_cost(input_tokens, output_tokens, model="nova-lite"):
    pricing = {
        "nova-lite":   {"input": 0.06,  "output": 0.24},
        "nova-pro":    {"input": 0.80,  "output": 3.20},
        "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }
    rates = pricing.get(model, {"input": 0, "output": 0})
    input_cost = (input_tokens / 1_000_000) * rates["input"]
    output_cost = (output_tokens / 1_000_000) * rates["output"]
    return input_cost + output_cost

print("Cost per single agent trace (800 input tokens, 200 output tokens)")
print("=" * 55)
print(f"{'Model':<16} {'Input':>8} {'Output':>8} {'Cost':>12}")
print("-" * 55)
for model in ["nova-lite", "nova-pro", "gpt-4o-mini", "gpt-4o"]:
    cost = estimate_cost(800, 200, model)
    print(f"{model:<16} {800:>8} {200:>8} ${cost:>10.6f}")

print()
print("Projected monthly cost at 10,000 agent calls:")
print("-" * 55)
for model in ["nova-lite", "nova-pro", "gpt-4o-mini", "gpt-4o"]:
    monthly = estimate_cost(800, 200, model) * 10_000
    print(f"{model:<16} {'':>18} ${monthly:>10.2f}")

print()
print("Model choice matters. Check LangSmith to see your actual token usage per trace.")

---

## Stage 3 — Debugging Agent Failures

Agents fail. Tools throw exceptions, the LLM picks the wrong tool, inputs get mangled. The question is not *if* your agent will fail — it is *how fast can you find and fix the problem?*

LangSmith turns debugging from guesswork into a systematic process. You find the failed trace, click into the failing step, read the inputs and outputs, and trace backwards to the root cause.

We will intentionally create agents that fail in different ways and then walk through how to diagnose each failure in the dashboard.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def divide_numbers(numerator: float, denominator: float) -> str:
    """Divide two numbers. Use when asked to divide, calculate ratios, or find percentages."""
    if denominator == 0:
        raise ValueError("Cannot divide by zero. Provide a non-zero denominator.")
    result = numerator / denominator
    return f"{numerator} / {denominator} = {result:.4f}"

@tool
def get_metric(metric_name: str) -> str:
    """Retrieve a metric value from the database. Available: revenue, costs, zero_metric, employees."""
    data = {
        "revenue": "1000000",
        "costs": "750000",
        "zero_metric": "0",
        "employees": "450",
    }
    value = data.get(metric_name.lower())
    if value is None:
        return f"ERROR: Metric '{metric_name}' not found. Available: {list(data.keys())}"
    return value

math_agent = create_agent(
    model="bedrock:us.amazon.nova-lite-v1:0",
    tools=[divide_numbers, get_metric],
    system_prompt=(
        "You are a financial calculator assistant. "
        "Use get_metric to retrieve values, then divide_numbers for calculations. "
        "If a tool returns an ERROR, explain the problem to the user."
    ),
    name="financial_calculator_agent"
)

print("Agent created: financial_calculator_agent")
print("Tools: divide_numbers (can raise ValueError), get_metric (can return ERROR)")

In [ ]:
print("=" * 60)
print("Scenario 1: Successful calculation")
print("=" * 60)
result1 = math_agent.invoke({
    "messages": [{"role": "user", "content": "What is revenue divided by costs?"}]
})
print(f"Query:    What is revenue divided by costs?")
print(f"Response: {result1['messages'][-1].content}")

print("\n" + "=" * 60)
print("Scenario 2: Division by zero (will trigger ValueError)")
print("=" * 60)
try:
    result2 = math_agent.invoke({
        "messages": [{"role": "user", "content": "What is revenue divided by zero_metric?"}]
    })
    print(f"Query:    What is revenue divided by zero_metric?")
    print(f"Response: {result2['messages'][-1].content}")
except Exception as e:
    print(f"Query:    What is revenue divided by zero_metric?")
    print(f"Error:    {e}")

print("\n" + "=" * 60)
print("Scenario 3: Requesting a metric that does not exist")
print("=" * 60)
result3 = math_agent.invoke({
    "messages": [{"role": "user", "content": "Get the profit margin metric."}]
})
print(f"Query:    Get the profit margin metric.")
print(f"Response: {result3['messages'][-1].content}")

print("\nAll failure traces are now in LangSmith. Let's learn how to debug them.")

### The Debugging Workflow

When something goes wrong, follow these six steps in LangSmith:

1. **Find the failed trace** — Look for red/error indicators in the trace list. Filter by status or time range to narrow things down.

2. **Identify the failure point** — Expand the trace tree. Find the step marked as failed. It could be an LLM call, a tool call, or the parent chain itself.

3. **Examine the inputs** — What was sent to the failing step? Was the input malformed or unexpected? Did a previous step produce bad data?

4. **Check the output/error** — Read the actual error message. Is it a code exception or a graceful error return? What does the error tell you about the root cause?

5. **Trace backwards** — If a tool received bad input, check the LLM call that invoked it. Why did the LLM pass those arguments? Was the tool description unclear? Was the system prompt missing guidance?

6. **Fix and verify** — Apply the fix (improve the tool, update the prompt, add error handling). Run the same query again and compare the new trace to the old one.

> **The pattern:** Failure is always a chain. The error you *see* (tool crash) is rarely the error you need to *fix* (unclear tool description, missing validation, bad prompt). Traces let you walk the chain backward to the real root cause.

### Wrong Tool Selection

Sometimes the agent doesn't crash — it just calls the *wrong* tool. This is harder to catch without traces because the agent still returns a response; it's just based on the wrong data.

The root cause is almost always **vague tool descriptions**. If the LLM can't tell your tools apart, it guesses. Let's demonstrate this by creating three tools with identical descriptions.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_customers(name: str) -> str:
    """Search by name."""
    return f"Customer: {name} — Email: {name.lower()}@example.com"

@tool
def search_products(name: str) -> str:
    """Search by name."""
    return f"Product: {name} — Price: $99.99, Stock: 42 units"

@tool
def search_orders(name: str) -> str:
    """Search by name."""
    return f"Orders for {name}: #1234 (Shipped), #5678 (Processing)"

confused_agent = create_agent(
    model="bedrock:us.amazon.nova-lite-v1:0",
    tools=[search_customers, search_products, search_orders],
    system_prompt="You are a customer service assistant. Use the appropriate search tool.",
    name="confused_service_agent"
)

print("Agent created with THREE tools that all say 'Search by name.'")
print("The LLM has no way to tell them apart — watch what happens.\n")

print("=" * 60)
print("Query: Find John's orders")
print("=" * 60)
result = confused_agent.invoke({
    "messages": [{"role": "user", "content": "Find John's orders"}]
})
print(f"Response: {result['messages'][-1].content}")

print("\n" + "=" * 60)
print("Query: What is the email for customer Sarah?")
print("=" * 60)
result = confused_agent.invoke({
    "messages": [{"role": "user", "content": "What is the email for customer Sarah?"}]
})
print(f"Response: {result['messages'][-1].content}")

print("\nCheck LangSmith: did the agent pick the RIGHT tool each time?")
print("If not, that is a tool-description problem — the fix is better descriptions.")

### Common Debugging Patterns

| Problem | What You See in the Trace | How to Fix It |
|---------|---------------------------|---------------|
| **Tool returns error** | Exception or `ERROR` string in tool output node | Add input validation or error handling inside the tool |
| **Wrong tool called** | LLM output shows `tool_calls` to the wrong function | Improve tool descriptions so each one is clearly distinct |
| **Agent ignores tools** | LLM output has no `tool_calls` at all | Update the system prompt to emphasize tool usage |
| **Wrong tool arguments** | Tool input has malformed or unexpected values | Fix parameter types, add descriptions to tool parameters |
| **Infinite tool loop** | Same tool called repeatedly in the trace tree | Add exit conditions, limit iterations, or improve the prompt |
| **Token limit exceeded** | Trace shows truncation or a context-length error | Summarize conversation history, shorten tool responses |

### The Fix for "Wrong Tool Selected"

Compare vague vs. specific tool descriptions:

```python
# Bad — all three look identical to the LLM
@tool
def search_customers(name: str) -> str:
    """Search by name."""

@tool
def search_products(name: str) -> str:
    """Search by name."""

# Good — each description explains WHEN to use this tool
@tool
def search_customers(name: str) -> str:
    """Look up a customer record by name. Use for customer info, emails, or accounts."""

@tool
def search_products(name: str) -> str:
    """Look up a product by name. Use for pricing, stock levels, or product details."""
```

The LLM reads tool descriptions to decide which tool to call. Vague descriptions lead to wrong choices. Specific descriptions lead to correct choices. LangSmith traces show you *which tool was picked* so you can verify the fix worked.

---

## Key Takeaways

1. **Three env vars enable full observability** — `LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, and `LANGSMITH_PROJECT`. No code changes needed once they are set.

2. **Traces are hierarchical** — every agent run produces a tree of LLM calls, tool calls, and state changes with timing and token data at every node.

3. **The dashboard is your debugging HQ** — filter by status, click into nodes, compare traces side by side. Stop guessing; start inspecting.

4. **Token usage = cost** — multi-tool traces consume more tokens because each LLM call includes the full conversation history. Monitor and optimize.

5. **Debug backwards** — the error you see (tool crash) is rarely the error you fix (bad prompt, unclear description). Trace the chain from failure to root cause.

6. **Tool descriptions drive tool selection** — vague descriptions cause wrong tool calls. LangSmith traces prove it; better descriptions fix it.

---

## Up Next: Tuesday — Vector Databases & Pinecone

Now that you can observe and debug your agents, we will give them **long-term memory**. On Tuesday we introduce **vector databases** — a way to store and retrieve information by *meaning* rather than keywords. You will set up **Pinecone**, embed documents, and build agents that can search a knowledge base to answer questions grounded in real data.

The progression: you built agents (Week 4) → you can observe them (today) → you will give them memory (tomorrow).